# Demographics Between Cohorts

This notebook creates a manuscript-style demographics table comparing training, internal-test, and external-test cohorts. The p values compare internal test vs training and external test vs training.

In [1]:
# ============================================================
# 1. Imports
# ============================================================

import os
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

In [2]:
# ============================================================
# 2. Settings
# ============================================================

patient_list_out_dir = '/host/e/D/Data/Habitats/Jishuitan/Patient_lists'
results_out_dir = '/host/d/projects/Habitats/results'
os.makedirs(results_out_dir, exist_ok=True)

processed_clinical_variables_path = os.path.join(
    patient_list_out_dir,
    'image_label_info_set123_clinical_variables_processed.xlsx',
)

split_file = os.path.join(
    patient_list_out_dir,
    'image_label_info_set123_5fold_prognosis_random0.xlsx',
)

between_cohort_table_path = os.path.join(
    results_out_dir,
    'demographics_between_cohort_table_prognosis.xlsx',
)

label_col = 'Prognosis_label'
lesion_site_levels = ['Femur', 'Tibia and fibula', 'Others']

print('processed_clinical_variables_path:', processed_clinical_variables_path)
print('split_file:', split_file)
print('between_cohort_table_path:', between_cohort_table_path)

processed_clinical_variables_path: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123_clinical_variables_processed.xlsx
split_file: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123_5fold_prognosis_random0.xlsx
between_cohort_table_path: /host/d/projects/Habitats/results/demographics_between_cohort_table_prognosis.xlsx


In [3]:
# ============================================================
# 3. Load processed clinical variables and split table
# ============================================================

clinical_df = pd.read_excel(
    processed_clinical_variables_path,
    dtype={'Patient_index': str},
)
clinical_df['Patient_set'] = clinical_df['Patient_set'].astype(str)
clinical_df['Patient_index'] = clinical_df['Patient_index'].astype(str)

split_df = pd.read_excel(
    split_file,
    dtype={'Patient_index': str},
)
split_df['Patient_set'] = split_df['Patient_set'].astype(str)
split_df['Patient_index'] = split_df['Patient_index'].astype(str)

split_keep_cols = ['Patient_set', 'Patient_index', 'split', 'fold', label_col]
missing_split_cols = [col for col in split_keep_cols if col not in split_df.columns]
if missing_split_cols:
    raise KeyError(f'Missing split columns: {missing_split_cols}')

clinical_for_merge = clinical_df.drop(columns=[label_col], errors='ignore').copy()

clinical_split_df = clinical_for_merge.merge(
    split_df[split_keep_cols],
    on=['Patient_set', 'Patient_index'],
    how='left',
)

if clinical_split_df['split'].isna().any():
    missing_cases = clinical_split_df.loc[
        clinical_split_df['split'].isna(),
        ['Patient_set', 'Patient_index'],
    ]
    raise RuntimeError(f'Some clinical cases were not found in split file:\n{missing_cases}')

print('clinical_split_df shape:', clinical_split_df.shape)
print('\nSplit counts:')
display(clinical_split_df['split'].value_counts(dropna=False))
print('\nLabel counts by split:')
display(pd.crosstab(clinical_split_df['split'], clinical_split_df[label_col]))

clinical_split_df shape: (348, 40)

Split counts:


train            188
internal test     96
external test     64
Name: split, dtype: int64


Label counts by split:


Prognosis_label,0,1
split,,
external test,47,17
internal test,72,24
train,139,49


In [4]:
# ============================================================
# 4. Variable lists and display names
# ============================================================

categorical_variables_all = [
    'Sex',
    'Lesion_site',
    'Pathologic_fracture',
]

continuous_variables_all = [
    'Age',
    'Height_at_visit',
    'Weight_at_visit',
    'BMI',
    'WBC',
    'HGB',
    'PLT',
    'CRP',
    'ALP',
    'Total_cholesterol',
    'Triglycerides',
    'LDL',
    'LDH',
    'PT',
    'APTT',
    'Fibrinogen',
    'D_dimer',
    'Tumor_AP_diameter_mm',
    'Tumor_longitudinal_diameter_mm',
    'Tumor_transverse_diameter_mm',
    'Tumor_volume_mm3',
]

categorical_variables = [col for col in categorical_variables_all if col in clinical_split_df.columns]
continuous_variables = [col for col in continuous_variables_all if col in clinical_split_df.columns]

variable_display_name_map = {
    'Age': 'Age (years)',
    'Height_at_visit': 'Height at visit (cm)',
    'Weight_at_visit': 'Weight at visit (kg)',
    'BMI': 'BMI (kg/m²)',
    'WBC': 'WBC (×10⁹/L)',
    'HGB': 'HGB (g/L)',
    'PLT': 'PLT (×10⁹/L)',
    'CRP': 'CRP (mg/L)',
    'ALP': 'ALP (IU/L)',
    'Total_cholesterol': 'Total cholesterol (mmol/L)',
    'Triglycerides': 'Triglycerides (mmol/L)',
    'LDL': 'LDL (mmol/L)',
    'LDH': 'LDH (IU/L)',
    'PT': 'PT (s)',
    'APTT': 'APTT (s)',
    'Fibrinogen': 'Fibrinogen (mg/dL)',
    'D_dimer': 'D-dimer (mg/L FEU)',
    'Tumor_AP_diameter_mm': 'Tumor AP diameter (mm)',
    'Tumor_longitudinal_diameter_mm': 'Tumor longitudinal diameter (mm)',
    'Tumor_transverse_diameter_mm': 'Tumor transverse diameter (mm)',
    # Original tumor volume was calculated in mm³. For reporting, convert values to cm³.
    'Tumor_volume_mm3': 'Tumor volume (cm³)',
}

def display_variable_name(variable):
    return variable_display_name_map.get(variable, variable)

print('Categorical variables used:', categorical_variables)
print('Continuous variables used:', continuous_variables)

Categorical variables used: ['Sex', 'Lesion_site', 'Pathologic_fracture']
Continuous variables used: ['Age', 'Height_at_visit', 'Weight_at_visit', 'BMI', 'WBC', 'HGB', 'PLT', 'ALP', 'Total_cholesterol', 'Triglycerides', 'LDL', 'LDH', 'PT', 'Fibrinogen', 'D_dimer', 'Tumor_AP_diameter_mm', 'Tumor_longitudinal_diameter_mm', 'Tumor_transverse_diameter_mm', 'Tumor_volume_mm3']


In [5]:
# ============================================================
# 5. Helper functions
# ============================================================

def collapse_lesion_site_series(series):
    def _collapse_one(x):
        if pd.isna(x):
            return np.nan
        x = str(x).strip()
        x_lower = x.lower().replace('_', ' ').replace('-', ' ')
        x_lower = ' '.join(x_lower.split())
        if x_lower == 'femur':
            return 'Femur'
        if x_lower in ['tibia', 'fibula', 'tibia and fibula', 'tibia fibula', 'tibia/fibula']:
            return 'Tibia and fibula'
        return 'Others'
    return series.map(_collapse_one)


def format_sig(value, sig=3):
    if pd.isna(value):
        return ''
    value = float(value)
    if value == 0:
        return '0'
    return f'{value:.{sig}g}'


def format_p_value(p):
    if pd.isna(p):
        return ''
    p = float(p)
    if p < 0.001:
        return '<0.001'
    return format_sig(p, sig=3)


def get_continuous_values(df, variable):
    values = pd.to_numeric(df[variable], errors='coerce').astype(float)
    if variable == 'Tumor_volume_mm3':
        values = values / 1000.0
    return values


def format_mean_sd(values):
    values = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    if len(values) == 0:
        return 'NA'
    mean = values.mean()
    sd = values.std(ddof=1)
    return f'{format_sig(mean, 3)} ± {format_sig(sd, 3)}'


def continuous_p_value(x0, x1):
    x0 = pd.to_numeric(pd.Series(x0), errors='coerce').dropna()
    x1 = pd.to_numeric(pd.Series(x1), errors='coerce').dropna()
    if len(x0) < 2 or len(x1) < 2:
        return np.nan

    normal0 = stats.shapiro(x0).pvalue > 0.05 if 3 <= len(x0) <= 5000 else False
    normal1 = stats.shapiro(x1).pvalue > 0.05 if 3 <= len(x1) <= 5000 else False

    if normal0 and normal1:
        return float(stats.ttest_ind(x0, x1, equal_var=False, nan_policy='omit').pvalue)
    return float(stats.mannwhitneyu(x0, x1, alternative='two-sided').pvalue)


def count_percent_summary(values, level):
    values = pd.Series(values).dropna()
    if len(values) == 0:
        return '0 (NA%)'
    if pd.api.types.is_numeric_dtype(values):
        match = pd.to_numeric(values, errors='coerce') == float(level)
    else:
        match = values.astype(str) == str(level)
    n = int(match.sum())
    pct = n / len(values) * 100.0
    return f'{n} ({format_sig(pct, 3)}%)'


def categorical_global_p_value(train_values, compare_values):
    train_values = pd.Series(train_values).dropna()
    compare_values = pd.Series(compare_values).dropna()
    if len(train_values) == 0 or len(compare_values) == 0:
        return np.nan

    all_levels = sorted(set(train_values.astype(str)).union(set(compare_values.astype(str))))
    table = []
    for level in all_levels:
        table.append([
            int((train_values.astype(str) == level).sum()),
            int((compare_values.astype(str) == level).sum()),
        ])
    table = np.asarray(table, dtype=int)

    if table.shape[0] < 2:
        return np.nan

    try:
        chi2, p, dof, expected = stats.chi2_contingency(table)
        if table.shape == (2, 2) and np.any(expected < 5):
            return float(stats.fisher_exact(table)[1])
        return float(p)
    except Exception:
        return np.nan

In [6]:
# ============================================================
# 6. Build between-cohort demographics table
# ============================================================

if 'Lesion_site' in clinical_split_df.columns:
    clinical_split_df['Lesion_site'] = collapse_lesion_site_series(clinical_split_df['Lesion_site'])

analysis_groups = {
    'training': clinical_split_df[clinical_split_df['split'] == 'train'].copy(),
    'internal_test': clinical_split_df[clinical_split_df['split'] == 'internal test'].copy(),
    'external_test': clinical_split_df[clinical_split_df['split'] == 'external test'].copy(),
}

for dataset_name, df_group in analysis_groups.items():
    print('\n============================================================')
    print('Dataset:', dataset_name, 'n =', df_group.shape[0])
    print(df_group[label_col].value_counts(dropna=False).sort_index())

rows = []


def add_continuous_row(variable):
    train_values = get_continuous_values(analysis_groups['training'], variable)
    internal_values = get_continuous_values(analysis_groups['internal_test'], variable)
    external_values = get_continuous_values(analysis_groups['external_test'], variable)

    p_internal = continuous_p_value(train_values, internal_values)
    p_external = continuous_p_value(train_values, external_values)

    rows.append({
        'Variable': display_variable_name(variable),
        'Level': '',
        'Variable_type': 'continuous',
        'Training': format_mean_sd(train_values),
        'Internal test': format_mean_sd(internal_values),
        'p: internal test vs training': format_p_value(p_internal),
        'External test': format_mean_sd(external_values),
        'p: external test vs training': format_p_value(p_external),
    })


def add_categorical_rows(variable, display_name, levels, level_display=None):
    if variable not in categorical_variables:
        return
    level_display = level_display or {level: str(level) for level in levels}

    train_values = analysis_groups['training'][variable]
    internal_values = analysis_groups['internal_test'][variable]
    external_values = analysis_groups['external_test'][variable]

    p_internal = categorical_global_p_value(train_values, internal_values)
    p_external = categorical_global_p_value(train_values, external_values)

    rows.append({
        'Variable': display_name,
        'Level': '',
        'Variable_type': 'categorical',
        'Training': '',
        'Internal test': '',
        'p: internal test vs training': format_p_value(p_internal),
        'External test': '',
        'p: external test vs training': format_p_value(p_external),
    })

    for level in levels:
        rows.append({
            'Variable': display_name,
            'Level': level_display.get(level, str(level)),
            'Variable_type': 'categorical_level',
            'Training': count_percent_summary(train_values, level),
            'Internal test': count_percent_summary(internal_values, level),
            'p: internal test vs training': '',
            'External test': count_percent_summary(external_values, level),
            'p: external test vs training': '',
        })


add_categorical_rows(
    'Sex',
    'Sex',
    ['Female', 'Male'],
    {'Female': 'Female', 'Male': 'Male'},
)
add_categorical_rows(
    'Lesion_site',
    'Tumor location',
    lesion_site_levels,
    {'Femur': 'Femur', 'Tibia and fibula': 'Tibia and fibula', 'Others': 'Others'},
)
add_categorical_rows(
    'Pathologic_fracture',
    'Pathologic fracture',
    [0, 1],
    {0: 'No', 1: 'Yes'},
)

for var in continuous_variables:
    add_continuous_row(var)

between_cohort_df = pd.DataFrame(rows)

ordered_cols = [
    'Variable',
    'Level',
    'Variable_type',
    'Training',
    'Internal test',
    'p: internal test vs training',
    'External test',
    'p: external test vs training',
]
between_cohort_df = between_cohort_df[ordered_cols]

display(between_cohort_df)


Dataset: training n = 188
0    139
1     49
Name: Prognosis_label, dtype: int64

Dataset: internal_test n = 96
0    72
1    24
Name: Prognosis_label, dtype: int64

Dataset: external_test n = 64
0    47
1    17
Name: Prognosis_label, dtype: int64


,Variable,Level,Variable_type,Training,Internal test,p: internal test vs training,External test,p: external test vs training
0,Sex,,categorical,,,0.145,,0.816
1,Sex,Female,categorical_level,83 (44.1%),33 (34.4%),,30 (46.9%),
2,Sex,Male,categorical_level,105 (55.9%),63 (65.6%),,34 (53.1%),
3,Tumor location,,categorical,,,0.00466,,0.83
4,Tumor location,Femur,categorical_level,111 (59%),39 (40.6%),,35 (54.7%),
5,Tumor location,Tibia and fibula,categorical_level,48 (25.5%),42 (43.8%),,18 (28.1%),
6,Tumor location,Others,categorical_level,29 (15.4%),15 (15.6%),,11 (17.2%),
7,Pathologic fracture,,categorical,,,0.643,,0.372
8,Pathologic fracture,No,categorical_level,174 (92.6%),91 (94.8%),,62 (96.9%),
9,Pathologic fracture,Yes,categorical_level,14 (7.45%),5 (5.21%),,2 (3.12%),


In [7]:
# ============================================================
# 7. Save between-cohort demographics table
# ============================================================

between_cohort_df.to_excel(between_cohort_table_path, index=False)

print('Saved between-cohort demographics table:')
print(between_cohort_table_path)
print('Shape:', between_cohort_df.shape)
between_cohort_df.head(30)

Saved between-cohort demographics table:
/host/d/projects/Habitats/results/demographics_between_cohort_table_prognosis.xlsx
Shape: (29, 8)


,Variable,Level,Variable_type,Training,Internal test,p: internal test vs training,External test,p: external test vs training
0,Sex,,categorical,,,0.145,,0.816
1,Sex,Female,categorical_level,83 (44.1%),33 (34.4%),,30 (46.9%),
2,Sex,Male,categorical_level,105 (55.9%),63 (65.6%),,34 (53.1%),
3,Tumor location,,categorical,,,0.00466,,0.83
4,Tumor location,Femur,categorical_level,111 (59%),39 (40.6%),,35 (54.7%),
5,Tumor location,Tibia and fibula,categorical_level,48 (25.5%),42 (43.8%),,18 (28.1%),
6,Tumor location,Others,categorical_level,29 (15.4%),15 (15.6%),,11 (17.2%),
7,Pathologic fracture,,categorical,,,0.643,,0.372
8,Pathologic fracture,No,categorical_level,174 (92.6%),91 (94.8%),,62 (96.9%),
9,Pathologic fracture,Yes,categorical_level,14 (7.45%),5 (5.21%),,2 (3.12%),
